In [ ]:
#!pip install openpyxl -q


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import pandas as pd
import numpy as np # Pour np.nan utilisé dans le nettoyage

# ----------------------------------------------------------------------
# 1. FONCTION MODIFIÉE : explorer_arborescence
#    (Correction pour garantir que le nom du fichier est en Niveau_Niv)
# ----------------------------------------------------------------------

def explorer_arborescence(path_folder_input: str, 
                         Niv: int = 3, 
                         with_files: bool = False) -> pd.DataFrame:
    """
    Explore une arborescence et génère un DataFrame représentant la hiérarchie.

    Paramètres
    ----------
    path_folder_input : str
        Chemin du dossier racine à explorer.
    Niv : int
        Profondeur maximale à explorer (détermine le nombre de colonnes).
    with_files : bool
        Inclure ou non les fichiers dans la structure.
        
    Retourne
    -------
    pd.DataFrame
        Le DataFrame représentant l'arborescence (non nettoyé).
    """
    
    # Vérifier que le dossier existe
    if not os.path.isdir(path_folder_input):
        raise ValueError(f"Le dossier '{path_folder_input}' n'existe pas.")
    
    lignes = []

    def parcourir(repertoire, niveau_actuel, chemin_courant):
        """Fonction récursive pour parcourir l'arborescence"""
        if niveau_actuel > Niv:
            return
        
        try:
            elements = sorted(os.listdir(repertoire))
        except PermissionError:
            return
        
        # Le padding pour un dossier le place au niveau_actuel
        padding_size_dossier = Niv - niveau_actuel
        
        for elem in elements:
            chemin_elem = os.path.join(repertoire, elem)
            
            if os.path.isdir(chemin_elem):
                # Ajout de la ligne pour le DOSSIER :
                # [chemin_courant] + [nom_dossier] + [padding]
                lignes.append(chemin_courant + [elem] + [""]*padding_size_dossier)
                # Appel récursif
                parcourir(chemin_elem, niveau_actuel+1, chemin_courant + [elem])
            elif with_files:
                # --- CORRECTION POUR LES FICHIERS ---
                # Le fichier doit être placé dans la colonne la plus à droite (Niveau_Niv).
                nb_colonnes_chemin = len(chemin_courant)
                padding_files = Niv - nb_colonnes_chemin - 1
                
                # S'assurer qu'il y a de la place pour le fichier
                if padding_files >= 0:
                    # [chemin_courant] + [padding pour le décalage] + [nom_fichier à la fin]
                    lignes.append(chemin_courant + [""]*padding_files + [elem])
    
    # Lancer l'exploration
    # On commence à Niveau 1, sans chemin parent initial
    parcourir(path_folder_input, 1, [])
    
    # Construire le DataFrame
    colonnes = [f"Niveau_{i}" for i in range(1, Niv+1)]
    # Gère le cas où lignes est vide
    df = pd.DataFrame(lignes, columns=colonnes) if lignes else pd.DataFrame(columns=colonnes)
    
    return df

# ----------------------------------------------------------------------
# 2. FONCTION GLOBALE : explorer_arborescences_global
#    (Correction de la logique de nettoyage pour conserver tous les fichiers)
# ----------------------------------------------------------------------

def explorer_arborescences_global(folders_to_list: list[list], 
                                  path_folder_output: str, 
                                  output_filename: str = "Arborescences_Globales.xlsx") -> None:
    """
    Explore plusieurs arborescences de dossiers et consolide les résultats
    dans un seul fichier Excel, en alignant les colonnes et en nettoyant.

    Paramètres
    ----------
    folders_to_list : list[list]
        Liste des configurations. Chaque sous-liste doit contenir :
        [path_folder_input: str, Niv: int, with_files: bool].
    path_folder_output : str
        Dossier où sera sauvegardé le fichier Excel consolidé.
    output_filename : str
        Nom du fichier de sortie (par défaut : "Arborescences_Globales.xlsx").
    """
    
    if not folders_to_list:
        print("La liste des dossiers à explorer est vide.")
        return

    # 1. Déterminer le niveau maximum (Niv) pour l'alignement des colonnes
    try:
        niv_max_global = max(item[1] for item in folders_to_list)
    except IndexError:
        print("Erreur : La configuration des dossiers est incorrecte. Format attendu : [path, Niv, with_files].")
        return
        
    colonnes_finales = [f"Niveau_{i}" for i in range(1, niv_max_global + 1)]
    derniere_colonne = colonnes_finales[-1]
    
    all_dfs = []

    # 2. Itérer, explorer et aligner chaque DataFrame
    for config in folders_to_list:
        try:
            path_input, niv_local, with_files = config[0], config[1], config[2]
            
            df_local = explorer_arborescence(path_input, niv_local, with_files)
            
            # --- Étape d'Alignement ---
            
            # 2a. Ajouter les colonnes manquantes (si Niv_local < Niv_max_global)
            for i in range(niv_local + 1, niv_max_global + 1):
                df_local[f"Niveau_{i}"] = "" 
            
            # 2b. S'assurer que les colonnes sont dans le bon ordre
            df_local = df_local[colonnes_finales]
            
            all_dfs.append(df_local)

        except ValueError as e:
            print(f"Erreur (Dossier introuvable) lors de l'exploration de {config[0]} : {e}")
            continue
        except IndexError:
            print(f"Erreur de format de configuration pour {config[0]}. Chaque élément doit être [path, Niv, with_files].")
            continue
        except Exception as e:
            print(f"Une erreur inattendue s'est produite lors de l'exploration de {config[0]} : {e}")
            continue

    if not all_dfs:
        print("Aucune arborescence n'a pu être explorée avec succès. Fichier Excel non généré.")
        return

    # 3. Concaténer tous les DataFrames alignés
    final_df = pd.concat(all_dfs, ignore_index=True)

    # 4. Nettoyer le DataFrame consolidé (LOGIQUE DE NETTOYAGE RÉVISÉE)
    # Pour CONSERVER tous les fichiers, nous ne devons pas utiliser dropna(how='any').
    # Nous conservons uniquement les lignes où le dernier élément (fichier ou dossier)
    # est présent dans la dernière colonne (Niveau_max_global).
    
    print("\nAjustement du DataFrame : Conservation des lignes dont la dernière colonne est remplie (fichiers ou éléments de fin de chemin)...")
    
    # Conserver toutes les lignes où la dernière colonne n'est PAS vide
    df_clean = final_df[final_df[derniere_colonne] != ""].copy()
    
    # Remplacer les chaînes vides restantes par NaN pour un affichage propre dans Excel
    df_clean.replace("", np.nan, inplace=True) 

    final_df = df_clean
    
    # 5. Exporter vers Excel
    output_file = os.path.join(path_folder_output, output_filename)
    os.makedirs(path_folder_output, exist_ok=True)
    
    final_df.to_excel(output_file, index=False)
    
    print(f"\nFichier Excel global généré : {output_file}")
    

### Films

In [9]:
path_folder_output="C:/Users/FLORIAN/Desktop"

folders_to_list = [  
   ["H:/Vidéos/Films/" , 3  ,True ],  
   ["D:/Videos/Films/" , 3  ,True ]
]

explorer_arborescences_global(folders_to_list, path_folder_output)


Ajustement du DataFrame : Conservation des lignes dont la dernière colonne est remplie (fichiers ou éléments de fin de chemin)...

Fichier Excel global généré : C:/Users/FLORIAN/Desktop\Arborescences_Globales.xlsx


### Musique

peut être rajouter les sons de BO de films

In [2]:
path_folder_output="C:/Users/FLORIAN/Desktop"

folders_to_list = [  
   ["C:/Users/FLORIAN/Music/__QBDLX_Download" , 2  ,True ],  
   ["C:/Users/FLORIAN/Music/__Albums" , 2  ,False ],  
   ["E:/Musique/__Autres"  , 2  ,False ],  
   ["E:/Musique/__B.O"  , 2  ,False ],  
   ["E:/Musique/__CLASSIQUE"  , 2  ,False ],  
    ["E:/Musique/__COMPILS"  , 2  ,False ],  
]

explorer_arborescences_global(folders_to_list, path_folder_output)


Ajustement du DataFrame : Conservation des lignes dont la dernière colonne est remplie (fichiers ou éléments de fin de chemin)...

Fichier Excel global généré : C:/Users/FLORIAN/Desktop\Arborescences_Globales.xlsx


### NAS TrueNAS

In [2]:
path_folder_output="C:/Users/FLORIAN/Desktop"

folders_to_list = [
   ["M:/musiques/__Autres"  , 2  ,False ],  
   ["M:/musiques/__B.O"  , 1  ,False ],  
   ["M:/musiques/__CLASSIQUE"  , 1  ,False ],  
    ["M:/musiques/__COMPILS"  , 1  ,False ],  
]

explorer_arborescences_global(folders_to_list, path_folder_output)


Ajustement du DataFrame : Conservation des lignes dont la dernière colonne est remplie (fichiers ou éléments de fin de chemin)...

Fichier Excel global généré : C:/Users/FLORIAN/Desktop\Arborescences_Globales.xlsx


### ARE FranceTravail

In [3]:
path_folder_output="C:/Users/FLORIAN/Desktop"

folders_to_list = [  
   ["C:\\Users\\FLORIAN\\Documents\\ARE_france_travail" , 1  ,True ],  
]

explorer_arborescences_global(folders_to_list, path_folder_output)


Ajustement du DataFrame : Conservation des lignes dont la dernière colonne est remplie (fichiers ou éléments de fin de chemin)...

Fichier Excel global généré : C:/Users/FLORIAN/Desktop\Arborescences_Globales.xlsx


In [3]:
path_folder_output="C:/Users/FLORIAN/Desktop"

folders_to_list = [  
   ["C:\\Users\\FLORIAN\\Desktop\\Utilitaires\\Racourcis" , 1  ,True ],  
]

explorer_arborescences_global(folders_to_list, path_folder_output)


Ajustement du DataFrame : Conservation des lignes dont la dernière colonne est remplie (fichiers ou éléments de fin de chemin)...

Fichier Excel global généré : C:/Users/FLORIAN/Desktop\Arborescences_Globales.xlsx
